In [1]:
import numpy as np
import pandas as pd

import string, faiss

from transformers import AutoModelForSeq2SeqLM, AutoTokenizer, set_seed
from sentence_transformers import SentenceTransformer

In [2]:
set_seed(42)

In [3]:
df=pd.read_csv('data/exoplanets.csv', index_col=0)

In [4]:
df.head(3)

,pl_name,hostname,discoverymethod,disc_year,disc_facility,pl_orbper,pl_rade,pl_bmasse,pl_eqt,pl_orbsmax,pl_orbeccen,st_teff,st_rad,st_mass,sy_dist,sy_vmag,ra,dec,rowupdate
0,TOI-4495 b,TOI-4495,Transit,2026.0,Transiting Exoplanet Survey Satellite (TESS),2.566990,2.483000,7.7,1735.0,0.03957,0.078,6210.00,1.309000,1.247,132.214,9.531,286.362495,37.025990,2026-02-12
1,TOI-4511 b,TOI-4511,Transit,2026.0,Transiting Exoplanet Survey Satellite (TESS),10.980910,2.753049,NaN,NaN,NaN,NaN,5418.77,1.001450,NaN,121.832,10.537,49.305279,15.501727,2026-04-23
2,TOI-4513 b,TOI-4513,Transit,2026.0,Transiting Exoplanet Survey Satellite (TESS),9.760423,2.345876,NaN,NaN,NaN,NaN,5623.93,0.796628,NaN,102.609,10.696,18.090518,13.755080,2026-04-23


In [5]:
# replace spaces with "-" in names
df['pl_name'] = df['pl_name'].apply(lambda x: x.replace(" ", "-"))
df['hostname'] = df['hostname'].apply(lambda x: x.replace(" ", "-"))

In [6]:
# convert discovery year to int
df['disc_year']=df['disc_year'].astype('Int32')

In [7]:
# create verbose description of each entry
df['description'] = df.apply(lambda x: f"The planet {x.pl_name} orbits around the star {x.hostname}. It was discovered in {x.disc_year}. It has a mass of {x.pl_bmasse} Mass Earth Units. It has an orbital period of {x.pl_orbper} days. It has a radius of {x.pl_rade} Radius Earth Units. It has a temperature {x.pl_eqt} K", axis=1)
df['description'] = df['description'].apply(lambda x: x.replace(" nan ", " unknown "))

In [8]:
# create schematic summary of each entry
df['summary'] = df.apply(lambda x: f"name {x.pl_name} | host_name {x.hostname} | discovery_year {x.disc_year} | mass {x.pl_bmasse} | orbital_period {x.pl_orbper} | radius {x.pl_rade} | temperature {x.pl_eqt}", axis=1)
df['summary'] = df['summary'].apply(lambda x: x.replace(" nan ", " unknown "))

In [9]:
df.head(3)['description'].tolist()

['The planet TOI-4495-b orbits around the star TOI-4495. It was discovered in 2026. It has a mass of 7.7 Mass Earth Units. It has an orbital period of 2.56699 days. It has a radius of 2.483 Radius Earth Units. It has a temperature 1735.0 K',
 'The planet TOI-4511-b orbits around the star TOI-4511. It was discovered in 2026. It has a mass of unknown Mass Earth Units. It has an orbital period of 10.98090968596 days. It has a radius of 2.75304932 Radius Earth Units. It has a temperature unknown K',
 'The planet TOI-4513-b orbits around the star TOI-4513. It was discovered in 2026. It has a mass of unknown Mass Earth Units. It has an orbital period of 9.76042253684 days. It has a radius of 2.34587608 Radius Earth Units. It has a temperature unknown K']

In [10]:
df.head(3)['summary'].tolist()

['name TOI-4495-b | host_name TOI-4495 | discovery_year 2026 | mass 7.7 | orbital_period 2.56699 | radius 2.483 | temperature 1735.0',
 'name TOI-4511-b | host_name TOI-4511 | discovery_year 2026 | mass unknown | orbital_period 10.98090968596 | radius 2.75304932 | temperature nan',
 'name TOI-4513-b | host_name TOI-4513 | discovery_year 2026 | mass unknown | orbital_period 9.76042253684 | radius 2.34587608 | temperature nan']

In [11]:
## create and index vector embeddings
#st_model = SentenceTransformer('all-MiniLM-L6-v2')
#corpus = df['description'].tolist()
#embeddings = st_model.encode(corpus, convert_to_numpy=True, show_progress_bar=True)
#index = faiss.IndexFlatL2(embeddings.shape[1])
#faiss.normalize_L2(embeddings)
#index.add(embeddings)
#faiss.write_index(index, 'exoplanet.index')
#np.save('exoplanet.npy', embeddings)
#
##retrieve relevant information by semantic search
#def retrieve(query, k=5):
#    embedding = st_model.encode([query], convert_to_numpy=True, show_progress_bar=False)
#    faiss.normalize_L2(embedding)
#    dst, idx = index.search(embedding, k)
#
#    return df.iloc[idx[0]]['description'].tolist()

In [12]:
def jaccard_similarity(wordlist1, wordlist2):
    intersection = len(list(set(wordlist1).intersection(set(wordlist2))))
    union = len(list(set(wordlist1).union(set(wordlist2))))
    return float(intersection) / float(union)

def my_distance(a, b):
    # convert to lowercase and remove punctuation
    a = a.lower().translate(str.maketrans(dict.fromkeys(string.punctuation)))
    b = b.lower().translate(str.maketrans(dict.fromkeys(string.punctuation)))
    return jaccard_similarity(a.split(), b.split())

# retrieve relevant information by jaccard similarity search
def retrieve(query, k=3):
    dst = df['summary'].apply(lambda x: my_distance(query, x))
    idx = np.argsort(dst)[-k:]
    
    return df['summary'][idx].to_list()

In [13]:
retrieve("Any planet discovered in 2020?")

['name OGLE-2018-BLG-1700L-b | host_name OGLE-2018-BLG-1700L | discovery_year 2020 | mass 1400.0 | orbital_period unknown | radius unknown | temperature nan',
 'name KMT-2018-BLG-0029L-b | host_name KMT-2018-BLG-0029L | discovery_year 2020 | mass 7.59 | orbital_period unknown | radius unknown | temperature nan',
 'name OGLE-2012-BLG-0838L-b | host_name OGLE-2012-BLG-0838L | discovery_year 2020 | mass 53.1 | orbital_period unknown | radius unknown | temperature nan']

In [14]:
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-large")
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-large")

In [15]:
example = (f"Context:\n\n{df['summary'].to_list()[2]}\n\n"
            f"{df['summary'].to_list()[1]}\n\n"
            f"{df['summary'].to_list()[0]}\n\n"
            f"Question:\n\nWhat do you know about {df['pl_name'].to_list()[0]}?\n\n"
            f"Answer:\n\n{df['description'].to_list()[0]}\n\n"
           )

In [16]:
print(example)

Context:

name TOI-4513-b | host_name TOI-4513 | discovery_year 2026 | mass unknown | orbital_period 9.76042253684 | radius 2.34587608 | temperature nan

name TOI-4511-b | host_name TOI-4511 | discovery_year 2026 | mass unknown | orbital_period 10.98090968596 | radius 2.75304932 | temperature nan

name TOI-4495-b | host_name TOI-4495 | discovery_year 2026 | mass 7.7 | orbital_period 2.56699 | radius 2.483 | temperature 1735.0

Question:

What do you know about TOI-4495-b?

Answer:

The planet TOI-4495-b orbits around the star TOI-4495. It was discovered in 2026. It has a mass of 7.7 Mass Earth Units. It has an orbital period of 2.56699 days. It has a radius of 2.483 Radius Earth Units. It has a temperature 1735.0 K




In [17]:
def reply(question):

    context  = "\n\n".join(retrieve(question))

    prompt = f"Question:\n\n{question}\n\nContext:\n\n{context}\n\nAnswer:\n\n"

    complete_prompt = example + prompt

    #print(f"=== BEGIN PROMPT ===\n{complete_prompt}\n=== END PROMPT ===")

    inputs = tokenizer(complete_prompt, return_tensors="pt")

    ilen = inputs['input_ids'].shape[1]
    if ilen > 512:
        print(f"WARNING: input length is {ilen} > 512.")
    
    output = model.generate(
        **inputs,
        max_new_tokens=128,
        do_sample=True,
        temperature=0.20,
        top_p=0.95,
        no_repeat_ngram_size=3,
        eos_token_id=tokenizer.eos_token_id
    )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [18]:
reply("What is the radius of Kepler-1604-b and when was it discovered?")

'Kepler-1604-b was discovered in 2016 and has a radius of 1.41.'

In [19]:
df[df['pl_name']=='Kepler-1604-b']['summary'].to_list()

['name Kepler-1604-b | host_name Kepler-1604 | discovery_year 2016 | mass unknown | orbital_period 0.683684256 | radius 1.41 | temperature nan']

In [20]:
reply("Any planet discovered in 2010? What is its temperature?")

'The temperature of the planet HAT-P-16-b is 1566.0. It was discovered in 2010. It has a mass of unknown and an orbital period of unknown. Its radius is 14.68376472.'

In [21]:
df[df['pl_name']=='HAT-P-16-b']['summary'].to_list()

['name HAT-P-16-b | host_name HAT-P-16 | discovery_year 2010 | mass unknown | orbital_period unknown | radius 14.68376472 | temperature 1566.0']

In [22]:
reply("Any planets with a mass between 0.9 and 1.1?")

'Yes'

In [23]:
reply("Can you give me an example of a planet with mass between 0.9 and 1.1?")

'Kepler-59-b, discovered in 2012, has a mass of unknown. It has an orbital period of 11.8681707 and a radius of 1.1.'

In [24]:
# the llm cannot answer the previous question because it cannot retrieve enough information
retrieve("Can you give me an example of a planet with mass between 0.9 and 1.1?")

['name K2-156-b | host_name K2-156 | discovery_year 2018 | mass unknown | orbital_period 0.813143 | radius 1.1 | temperature nan',
 'name Kepler-1406-b | host_name Kepler-1406 | discovery_year 2016 | mass unknown | orbital_period 11.62905828 | radius 1.1 | temperature nan',
 'name Kepler-59-b | host_name Kepler-59 | discovery_year 2012 | mass unknown | orbital_period 11.8681707 | radius 1.1 | temperature nan']

In [25]:
retrieve("Can you give me an example of a planet discovered in 2014 with mass between 0.9 and 1.1?")

['name Kepler-392-c | host_name Kepler-392 | discovery_year 2014 | mass unknown | orbital_period 10.423118 | radius 1.1 | temperature nan',
 'name Kepler-374-c | host_name Kepler-374 | discovery_year 2014 | mass unknown | orbital_period 3.282807 | radius 1.1 | temperature nan',
 'name Kepler-102-b | host_name Kepler-102 | discovery_year 2014 | mass 1.1 | orbital_period 5.286965 | radius 0.46 | temperature 857.0']

In [26]:
reply("Can you give me an example of a planet discovered in 2014 with mass between 0.9 and 1.1?")

'Kepler-102-b was discovered in 2014 and has a mass of 1.1. It has an orbital period of 5.286965 and a radius of 0.46. Its host name is KepLER-102.'

In [27]:
reply("Which planets were discovered last year (in 2025)?")

'The OGLE-2015-BLG-1609L-b, KMT-2023-BLGN-0119L-B, and KMT-BLZ-0119-b were discovered in 2025.'

In [28]:
df[df['pl_name']=='KMT-2023-BLG-0119L-b']['summary'].to_list()

['name KMT-2023-BLG-0119L-b | host_name KMT-2023-BLG-0119L | discovery_year 2025 | mass 83.7 | orbital_period unknown | radius unknown | temperature nan']